# 06. Фінальний пайплайн, темпоральний бонус і сабміт

Беремо навчені моделі з 04 і 05 (на Kaggle додайте їхні виходи як Input) і перевіряємо на val:
1. зміну моделі при однаковому інференсі;
2. зміну даних і інференсу для однієї моделі: роздільність, TTA, тайли (як SAHI), поріг NMS;
3. ансамбль через Weighted Boxes Fusion;
4. темпоральне перерахування впевненості з компенсацією руху камери (бонус).

Інференс іде з оригінальних кадрів: зменшуємо тією самою `common.load_resized`, що й при підготовці train, а бокси повертаємо в оригінальні пікселі.
Прогноз робимо для всіх кадрів val (2846), щоб темпоральний метод мав сусідні кадри, як на тесті. Метрика рахується на розмічених кадрах з `annotations_eval.json` (кожен 2-й).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ostapmn/ucu_ai_hw_2.git"
if Path("/kaggle/working").exists() and not Path("common.py").exists():
    repo = Path("/kaggle/working/repo")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
    os.chdir(repo / "code")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, ".")
DEBUG = os.environ.get("HW2_DEBUG") == "1"

In [ ]:
import csv
import hashlib
import pickle
from itertools import groupby

import cv2
import numpy as np
import pandas as pd

import analysis as an
import common as c
import postprocess as pp
import yolo_utils as yu
from ultralytics import RTDETR, YOLO

yu.setup()
c.set_seed()
P = c.paths()
OUT = P.out / "final"
CACHE = OUT / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
GT = P.prep / "val" / "annotations_eval.json"

MODELS = {
    "yolo_1280": dict(sub="anchor_free", imgsz=1280, rtdetr=False),
    "yolo_p2": dict(sub="anchor_free", imgsz=960, rtdetr=False),
    "yolo_base": dict(sub="anchor_free", imgsz=960, rtdetr=False),
    "yolo_tal13": dict(sub="anchor_free", imgsz=960, rtdetr=False),
    "rtdetr_q500": dict(sub="query_based", imgsz=960, rtdetr=True),
    "rtdetr_q300": dict(sub="query_based", imgsz=960, rtdetr=True),
}
if DEBUG:
    for m in MODELS.values():
        m["imgsz"] = 320
WEIGHTS = {n: an.find_run_file(P, f"{m['sub']}/{n}/weights/best.pt") for n, m in MODELS.items()}
AVAILABLE = [n for n, w in WEIGHTS.items() if w is not None]
print("доступні моделі:", AVAILABLE)


def frames_val():
    ims = c.load_json(P.data / "val" / "annotations_coco.json")["images"]
    return sorted(ims, key=lambda i: (i["video"], i["frame_id"])), P.data / "val"


def frames_test():
    with open(P.data / "test_images.csv") as f:
        rows = list(csv.DictReader(f))
    fr = [dict(id=r["image_id"], file_name=r["file_name"], video=r["clip"], frame_id=int(r["frame"]),
               width=int(r["width"]), height=int(r["height"])) for r in rows]
    return sorted(fr, key=lambda i: (i["video"], i["frame_id"])), P.data


VAL, VAL_ROOT = frames_val()
TEST, TEST_ROOT = frames_test()
EVAL_IDS = [im["id"] for im in c.load_json(GT)["images"]]
print(len(VAL), "кадрів val,", len(EVAL_IDS), "з них розмічені для оцінки,", len(TEST), "кадрів test")

## Інструменти: прогноз з кешем, пайплайн, оцінка

In [ ]:
_models = {}


def get_model(name):
    if name not in _models:
        _models[name] = (RTDETR if MODELS[name]["rtdetr"] else YOLO)(WEIGHTS[name])
    return _models[name]


def predict_frames(frames, root, split, name, imgsz=None, tta=False, zoom=None, nms_iou=0.7, batch=8):
    """Детекції для кожного кадру (оригінальні пікселі). Результат кешується на диск.

    imgsz - довга сторона кадру на вході (за замовчуванням як у навчанні);
    zoom - тайли кадру, збільшеного в zoom разів відносно imgsz, плюс цілий кадр (див. pp.tiled_predict).
    """
    imgsz = imgsz or MODELS[name]["imgsz"]
    tta = tta and not MODELS[name]["rtdetr"]
    path = CACHE / f"{split}_{name}_{imgsz}_tta{int(tta)}_zoom{zoom}_nms{nms_iou}.pkl"
    if path.exists():
        return pickle.loads(path.read_bytes())
    model = get_model(name)

    def fn(imgs, size):
        return yu.predict_arrays(model, imgs, imgsz=size, iou=nms_iou, augment=tta, batch=batch)

    out = {}
    for i in range(0, len(frames), batch):
        chunk = frames[i:i + batch]
        loaded = [c.load_resized(root / f["file_name"], imgsz) for f in chunk]
        imgs = [np.ascontiguousarray(a[..., ::-1]) for a, _, _ in loaded]
        if zoom:
            for f, img, (_, sx, sy) in zip(chunk, imgs, loaded):
                big, bx, by = c.load_resized(root / f["file_name"], int(imgsz * zoom))
                out[f["id"]] = pp.tiled_predict(fn, np.ascontiguousarray(big[..., ::-1]), (bx, by), tile=imgsz,
                                                full=img, scale_full=(sx, sy))
        else:
            for f, d, (_, sx, sy) in zip(chunk, fn(imgs, imgsz), loaded):
                out[f["id"]] = (pp.unscale(d[0], (sx, sy)), d[1], d[2])
    path.write_bytes(pickle.dumps(out))
    return out


def motions_for(frames, root, split):
    """Матриці руху камери між сусідніми кадрами кожного кліпу (кеш)."""
    path = CACHE / f"{split}_motion.pkl"
    if path.exists():
        return pickle.loads(path.read_bytes())
    out = {}
    for _, group in groupby(frames, key=lambda f: f["video"]):
        est = pp.MotionEstimator()
        for f in group:
            out[f["id"]] = est(cv2.imread(str(root / f["file_name"])))
    path.write_bytes(pickle.dumps(out))
    return out


def temporal(dets, frames, root, split, motion=True, **kw):
    H = motions_for(frames, root, split) if motion else None
    out = {}
    for _, group in groupby(frames, key=lambda f: f["video"]):
        group = list(group)
        res = pp.temporal_rescore([dets[f["id"]] for f in group], [H[f["id"]] for f in group] if H else None, **kw)
        out.update(zip([f["id"] for f in group], res))
    return out


def pipeline(frames, root, split, cfg):
    """cfg: models=[...], predict={imgsz, tta, zoom, nms_iou}, wbf_iou, weights, temporal={...} або None."""
    parts = [predict_frames(frames, root, split, m, **cfg.get("predict", {})) for m in cfg["models"]]
    if len(parts) == 1:
        dets = parts[0]
    else:
        key = f"{split}_wbf_{'+'.join(cfg['models'])}_{sorted(cfg.get('predict', {}).items())}_{cfg.get('wbf_iou', 0.6)}_{cfg.get('weights')}"
        path = CACHE / f"{hashlib.md5(key.encode()).hexdigest()[:16]}.pkl"
        if path.exists():
            dets = pickle.loads(path.read_bytes())
        else:
            dets = {f["id"]: pp.wbf([p[f["id"]] for p in parts], f["width"], f["height"],
                                    iou=cfg.get("wbf_iou", 0.6), weights=cfg.get("weights"))
                    for f in frames}
            path.write_bytes(pickle.dumps(dets))
    if cfg.get("temporal"):
        dets = temporal(dets, frames, root, split, **cfg["temporal"])
    return dets


TABLE = {}


def evaluate(label, cfg, group):
    dets = pipeline(VAL, VAL_ROOT, "val", cfg)
    coco = [d for iid in EVAL_IDS for d in pp.to_coco(dets[iid], iid)]
    r = c.evaluate(GT, coco)
    TABLE[label] = {"група": group, **{k: r[k] for k in ("mAP", "AP50", "AP_small", "AP_medium", "AP_large")}, "cfg": cfg}
    an.log_result(P.results, f"pipe_{label}", "pipeline", r, config=str(cfg))
    print(f"{label:40s} mAP {r['mAP']:.4f}  AP50 {r['AP50']:.4f}  AP_small {r['AP_small']:.4f}")
    return r["mAP"]


def show(group):
    t = pd.DataFrame(TABLE).T
    return t[t["група"] == group].drop(columns=["група", "cfg"]).astype(float).round(4)

## 1. Зміна моделі (однаковий інференс)

In [ ]:
for name in AVAILABLE:
    evaluate(name, {"models": [name]}, "модель")
show("модель")

## 2. Зміна інференсу для найкращої моделі

Модель та сама, міняємо тільки вхідну роздільність, TTA (фліп і масштаби Ultralytics), тайли і поріг NMS.

In [ ]:
BEST = max(AVAILABLE, key=lambda n: TABLE[n]["mAP"])
size = MODELS[BEST]["imgsz"]
bigger = int(size * 4 / 3) // 32 * 32
variants = {
    f"{BEST} @{bigger}": {"predict": {"imgsz": bigger}},
    f"{BEST} тайли x1.5": {"predict": {"zoom": 1.5}},
    f"{BEST} тайли x2": {"predict": {"zoom": 2.0}},
    f"{BEST} NMS 0.6": {"predict": {"nms_iou": 0.6}},
    f"{BEST} NMS 0.5": {"predict": {"nms_iou": 0.5}},
}
if not MODELS[BEST]["rtdetr"]:
    variants[f"{BEST} TTA"] = {"predict": {"tta": True}}
for label, cfg in variants.items():
    evaluate(label, {"models": [BEST], **cfg}, "інференс")
show("інференс")

## 3. Ансамбль (WBF)

Дві найкращі моделі різних типів або роздільностей з тим самим найкращим інференсом, злиття через Weighted Boxes Fusion.

In [ ]:
inf = {k: v for k, v in TABLE.items() if v["група"] == "інференс"}
best_inf = max(inf, key=lambda k: inf[k]["mAP"]) if inf else None
BEST_PRED = inf[best_inf]["cfg"].get("predict", {}) if best_inf and inf[best_inf]["mAP"] > TABLE[BEST]["mAP"] else {}
ranked = sorted(AVAILABLE, key=lambda n: -TABLE[n]["mAP"])
if len(ranked) >= 2:
    pred = {k: v for k, v in BEST_PRED.items() if k != "imgsz"}
    for iou in (0.55, 0.65):
        evaluate(f"WBF {ranked[0]} + {ranked[1]} iou {iou}", {"models": ranked[:2], "predict": pred, "wbf_iou": iou}, "ансамбль")
    if len(ranked) >= 3:
        evaluate("WBF top-3 iou 0.6", {"models": ranked[:3], "predict": pred, "wbf_iou": 0.6}, "ансамбль")
    display(show("ансамбль"))
print("найкращий інференс для однієї моделі:", BEST_PRED)

## 4. Бонус: темпоральне перерахування впевненості

Детекції сусідніх кадрів зв'язуємо в треки (угорський алгоритм за IoU, той самий клас, рух камери компенсуємо афінною моделлю ORB+RANSAC).
Нова впевненість `s' = (1-w)·s + w·mean(треку)`, одиночні детекції множимо на `single_penalty`, пропуск в один кадр заповнюємо інтерпольованим боксом.
Сітку параметрів робимо малою: у val лише 7 клипів, великий перебір просто підженеться під них.

In [ ]:
base_label = max((k for k, v in TABLE.items() if v["група"] in ("модель", "інференс", "ансамбль")), key=lambda k: TABLE[k]["mAP"])
BASE_CFG = {k: v for k, v in TABLE[base_label]["cfg"].items() if k != "temporal"}
print("основа:", base_label)
grid = [dict(w=w, single_penalty=sp, iou_thr=0.3, motion=True) for w in (0.3, 0.5, 0.7) for sp in (0.6, 0.8, 1.0)]
grid += [dict(w=0.5, single_penalty=0.8, iou_thr=0.3, motion=False),
         dict(w=0.5, single_penalty=0.8, iou_thr=0.5, motion=True),
         dict(w=0.5, single_penalty=0.8, iou_thr=0.3, motion=True, fill_gap=False)]
for t in grid:
    evaluate(f"temporal {t}", {**BASE_CFG, "temporal": t}, "темпоральне")
show("темпоральне")

## 5. Вибір фінальних сабмітів і прогноз на test

Kaggle дозволяє обрати два фінальні сабміти. Беремо найкращий за val і найкращий без темпоральної частини, на випадок якщо темпоральні параметри підігнались під 7 val-клипів.

In [ ]:
summary = pd.DataFrame(TABLE).T
summary["mAP"] = summary["mAP"].astype(float)
ranked_rows = summary.sort_values("mAP", ascending=False)
FINAL_1 = ranked_rows.iloc[0]
others = ranked_rows[ranked_rows.index != FINAL_1.name]
no_temporal = others[others["група"] != "темпоральне"]
FINAL_2 = (no_temporal if FINAL_1["група"] == "темпоральне" and len(no_temporal) else others).iloc[0]
for k, row in (("submission_1", FINAL_1), ("submission_2", FINAL_2)):
    print(k, row.name, round(row.mAP, 4))
    test_dets = pipeline(TEST, TEST_ROOT, "test", row.cfg)
    preds = {f["id"]: pp.to_submission(test_dets[f["id"]], f["width"], f["height"]) for f in TEST}
    path = P.out / f"{k}.csv"
    c.write_submission(preds, P.data / "test_images.csv", path)
    print("рядків:", c.check_submission(path, P.data / "test_images.csv"), "->", path)
    c.save_json({"val_label": row.name, "val_mAP": row.mAP, "cfg": row.cfg}, P.out / f"{k}.json")

## Висновки

<!-- Заповнюється після запуску: окремо ефект зміни моделі (розділ 1) і зміни даних та інференсу (розділи 2-3); бонус: що дає темпоральна частина і рух камери (розділ 4), на яких клипах і класах; які два сабміти вибрано і чому (val і public LB). -->